# Behind the scenes: Day One
### -> How the data was downloaded

In [3]:
# imports required for this notebook

from pathlib import Path
import requests
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from workshop_utils import RAW_DIR, PROCESSED_DIR
from workshop_utils.data_utils import launch_ARCO_ERA5, load_lsm

### Berkeley Earth Surface Temperature (BEST) Dataset

In [ ]:
url = "https://berkeley-earth-temperature.s3.us-west-1.amazonaws.com/Global/Land_and_Ocean_complete.txt"
out_path = RAW_DIR / "berkeley_earth_global_temp.txt"

r = requests.get(url)

# uncomment to download:
#out_path.write_text(r.text)

416532

In [ ]:
# Define the columns based on Berkeley Earth's documentation
columns = [
    "Year", "Month", "Anomaly_Monthly", "Unc_Monthly",
    "Anomaly_Annual", "Unc_Annual", "Anomaly_FiveYear", "Unc_FiveYear",
    "Anomaly_TenYear", "Unc_TenYear", "Anomaly_TwentyYear", "Unc_TwentyYear"
]

df = pd.read_csv(
    RAW_DIR / "berkeley_earth_global_temp.txt",
    comment="%",              # Ignores all the header text lines starting with %
    sep=r"\s+",              # Handles variable whitespace spacing between numbers
    header=None,             # The file doesn't have a standard row header
    names=columns,           # Assigns our clean column names
    nrows=2100               # Optional: Reads only up to the first data block
)

df.head()

,Year,Month,Anomaly_Monthly,Unc_Monthly,Anomaly_Annual,Unc_Annual,Anomaly_FiveYear,Unc_FiveYear,Anomaly_TenYear,Unc_TenYear,Anomaly_TwentyYear,Unc_TwentyYear
0,1850,1,-0.753,0.365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1850,2,-0.202,0.416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1850,3,-0.367,0.373,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1850,4,-0.596,0.324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1850,5,-0.619,0.267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### NASA GISTEMP Surface Temperature Dataset

In [ ]:
gistemp_url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
gistemp_raw = RAW_DIR / "gistemp_global.csv"

# uncomment to download:
#gistemp_raw.write_text(requests.get(gistemp_url).text)

12878

In [ ]:
gistemp = pd.read_csv(gistemp_raw, skiprows=1)

# melt monthly columns into long format
gistemp_long = gistemp.melt(
    id_vars=["Year"],
    var_name="month",
    value_name="temp_anomaly"
)

gistemp_yearly = gistemp_long.groupby("Year")["temp_anomaly"].mean().reset_index()

#gistemp_yearly.to_csv(PROCESSED / "gistemp_global_yearly.csv", index=False)

### Philippines temperature and precipitation
Here, we use ERA5 data to specifically zoom into the Philippines and Bacolod. Below, ERA5 reanalysis data is downloaded for the entire region year by year, with a temperature recording 4 times per day (00, 06, 12, and 18 UTC), and daily precipitation sums.

In [ ]:
# actually, let me go ahead and download from Copernicus API
# bounding box for the Philippines (roughly):
N_lat, S_lat, W_lon, E_lon = 21.25, 4.5, 114, 126.75

from workshop_utils.download import download_era5_copernicus

# download in single year increments
for year in range(1940, 2025):
    download_era5_copernicus(
        variable=["2m_temperature"],
        year=[str(year)],
        month=[f"{i:02d}" for i in range(1, 13)],
        day=[f"{i:02d}" for i in range(1, 32)],
        time=[f"{i:02d}:00" for i in range(0, 24, 6)],
        area=[N_lat, W_lon, S_lat, E_lon],
        output_path=RAW_DIR / "ERA5" / "2m_temperature" / f"t2m_philippines_{year}.nc",
    )

### CHIRPS Precipitation Dataset

Might not use this, actually. Let's keep it in for now.

In [ ]:
url = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/netcdf/chirps-v2.0.monthly.nc"
ds = xr.open_dataset(url)
# ...

### Mauna Loa CO2 record -- NOAA Global Monitoring Laboratory

In [ ]:
url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
co2_raw = RAW_DIR / "mauna_loa_co2.csv"

# uncomment to download:
#co2_raw.write_text(requests.get(url).text)

38600

### ENSO (Niño3.4 Index) -- NOAA Physical Sciences Laboratory

In [3]:
# KNMI (Dutch Weather Agency) hosts a "Climate Explorer" tool that allows you do download various climate indices, including ENSO (El Niño Southern Oscillation) data. Here's how you can download and process the ENSO data to get a yearly average of the NINO3.4 index:

url = "https://climexp.knmi.nl/data/iersst_nino3.4a_rel.nc"
ds = xr.open_dataset(url,
                     engine="h5netcdf",
                     decode_times=False)
ds

<xarray.Dataset> Size: 17kB
Dimensions:   (time: 2076)
Coordinates:
  * time      (time) float32 8kB 0.0 1.0 2.0 ... 2.073e+03 2.074e+03 2.075e+03
Data variables:
    Nino3.4r  (time) float32 8kB ...
Attributes: (12/43)
    title:                      Nino3.4 index based on NOAA ERSSTv5 (in situ ...
    description:                NINO3.4 rel
    scripturl01:                http://climexp.knmi.nl/getindices.cgi?STATION...
    file:                       ersstv5.nc
    cdi:                        Climate Data Interface version 2.5.3 (https:/...
    source:                     In situ data: ICOADS R3.0 before 2015, NCEP i...
    ...                         ...
    time_coverage_end:          2026-04-15
    mask:                       ersstv5_nino3.4_mask.nc
    climexp_url:                https://climexp.knmi.nl/getindices.cgi?NCDCDa...
    scripturl02:                http://climexp.knmi.nl/dat2nc.cgi?id=someone@...
    history:                     2026-06-07 14:27:50 ./bin/dat2nc data/iersst...
    Conventions:                CF-1.0

### Aerosol Optical Depth (AOD, proxy for volcanic activity) -- NASA GISS
Should appear as large spikes:
- 1963  Agung
- 1982  El Chichón
- 1991  Pinatubo

In [ ]:
url = "https://data.giss.nasa.gov/modelforce/strataer/data/tau_reff_Sato-Lacis.nc"

### Total Solar Irradiance (TSI)
Source?

In [ ]:
year = 1875
url = f"https://www.ncei.noaa.gov/data/total-solar-irradiance/access/monthly/tsi_v03r00_monthly_s{year}01_e{year}12_c20240831.nc"

### Orbital forcing (Milankovitch cycles)
Just use a visual from the web.

### Köppen-Geiger climate maps from Beck et al.

In [ ]:
url = "https://doi.org/10.6084/m9.figshare.21789074"